Import necessary libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
import json
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

/tmp/ipykernel_483673/137333164.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Load required documents

In [2]:
loader = PyPDFLoader("govt_law_chpt_10.pdf")
pdf_pages = loader.load()
print(len(pdf_pages))

73


In [3]:
with open("govt_law_chpt_10_structure.json", "r") as f:
    index_data = json.load(f)

Flatten Hierarchical JSON to Langchain Documents while storing metadeta

In [4]:
index_docs = []

In [5]:
def process_node(node, parent=None):

    index_docs.append(
        Document(
            page_content=f"""
            Title: {node.get('title','')}
            Summary: {node.get('summary','')}
            """,
            metadata={
                "node_id": node.get("node_id"),
                "title": node.get("title"),
                "parent": parent,
                "start_page": node.get("start_index"),
                "end_page": node.get("end_index")
            }
        )
    )

    for child in node.get("nodes", []):
        process_node(child, node.get("title"))

In [6]:
for node in index_data["structure"]:
    process_node(node)

print(len(index_docs))

363


BM25 retriever

In [8]:
section_retriever = BM25Retriever.from_documents(index_docs)
section_retriever.k = 5

In [9]:
query = "What is the purpose of the Act?"

sections = section_retriever.invoke(query)

for s in sections:
    print(s.metadata["title"])

The purpose of the Act
Section 1-1. The purpose of the Act
Section 1-4. Undertakings with no employees, etc.
Section 1-5. Work performed at the home of the employee or employer
Section 4-1. General requirements regarding the working environment
